# Lektion 01 - Introduktion til AI-agenter

Velkommen til den første lektion i **AI-agenter for begyndere**-kurset!

En **AI-agent** er et program, der bruger en stor sprogmodel (LLM) som sin ræsonneringsmotor og kan tage *handlinger* i den virkelige verden — kalde API'er, forespørge databaser eller køre kode — for at nå et mål på vegne af en bruger.

I denne notesbog vil du bygge din første agent: en **rejseagent**, der anbefaler feriedestinationer. Undervejs vil du lære at:

1. Oprette forbindelse til Azure AI Foundry Agent Service ved hjælp af **Microsoft Agent Framework**.
2. Give agenten et **værktøj** — en simpel Python-funktion, den kan kalde.
3. Køre agenten og inspicere dens svar.
4. Streame agentens svar token-for-token.


## Setup

Før du kører denne notesbog, skal du sikre dig, at du har:

1. **Et Azure AI Foundry-projekt** med en implementeret chatmodel (fx `gpt-4o-mini`).
2. **Logget ind med Azure CLI** — kør `az login` i din terminal.
3. **Indstillet de nødvendige miljøvariabler:**
   - `AZURE_AI_PROJECT_ENDPOINT` — dit Azure AI Foundry-projekts endepunkt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — navnet på din implementerede model.

Cellen nedenfor installerer de Python-pakker, du skal bruge.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Oprettelse af din første agent

En agent har brug for to ting:

- **Instruktioner**, der fortæller den *hvem den er* og *hvordan den skal opføre sig* (en systemprompt).
- **Værktøjer** — Python-funktioner dekoreret med `@tool`, som agenten kan kalde for at hente information eller udføre handlinger.

Nedenfor definerer vi et enkelt værktøj, der returnerer en liste over populære feriedestinationer. Agenten vil bruge dette værktøj, når en bruger beder om rejseanbefalinger.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming-svar

For en mere interaktiv oplevelse kan du **streame** agentens svar. I stedet for at vente på det fulde svar, leverer agenten tekststykker, efterhånden som de genereres. Dette er især nyttigt i chatteinterfacer, hvor du ønsker at vise output i realtid.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Resumé

I denne lektion lærte du hvordan man:

- **Opretter en udbyder** der opretter forbindelse til Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Definerer et værktøj** ved hjælp af `@tool` dekoratoren, så agenten kan kalde dine Python-funktioner.
- **Kører agenten** med en brugermeddelelse og udskriver dens svar.
- **Streamer svar** for output i realtid.

I næste lektion vil vi udforske agentiske rammer mere i dybden og lære, hvordan man giver agenter mere kraftfulde værktøjer og evner til flerstegs ræsonnering.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, bedes du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det oprindelige dokument på dets modersmål bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der måtte opstå som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
